# Beamtime on LoKI: starter notebook

Your proposal is just below (folded), with the buttons *Open NICOS* and *Sample builder*. This notebook only gives you the tools; what to measure and how to analyze it is
up to you. Keep notes of your choices: they are part of your report.

**Tools**
- `rows`: your logbook (one line per count: file, kind, title, settings);
- `red.reduce(sample, background=..., empty_beam=..., banks=[...])`: $I(Q)$ per detector bank, with the official ESS
  software (esssans); `red.combine(result, thickness_cm)`: banks merged into rear/middle/front, in cm⁻¹;
- `la.guinier(q, I, dI)`, `la.sphere_fit(q, I, dI)`: Guinier analysis and the sphere model of sasmodels (SasView);
- `h5py` to look inside a NeXus file; `scipy.optimize` for your own fits.

**Your proposal** (click to open it):

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))
import nb_tools
nb_tools.proposal("loki/exercises/silica_spheres/beamtime.md")

## 1. Your sample

The sample changer of LoKI holds the samples of the library (in NICOS, `sample_info()` describes the one mounted). For your own question, design a sample with the sample builder: particles of the kinds you choose (spheres, cylinders, core-shell spheres, as many kinds as you want) in a H2O/D2O solvent. It is saved with your proposal and appears on the sample changer of NICOS: `move(sample_holder, 'its_name')`.

In [ ]:
nb_tools.sample_builder("loki", "3001")

## 2. Your measurements, in NICOS

1. Tab *Experiment*: give the *Proposal ID* of your proposal, then **Apply** (first: your own samples are kept with your proposal, NICOS finds them only then).
2. **Mount your sample** (tab *Instrument interaction*, command line): `move(sample_holder, 'its_name')`, the short name of a sample of the library (e.g. `silica_spheres`, `d2o`, `empty_beam`) or of the one you made in part 1. `sample_info()` describes the sample mounted; for a name NICOS does not know, it lists the samples available. Every count measures the sample mounted at that moment.
3. **Record**: `with nexusfile_open('my title'): count(t=60)`; your files go to the folder of your proposal (below). The measurements and their counting times are your decisions: your budget is in your proposal.

In [ ]:
nb_tools.nicos("loki")

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))     # the modules of the LoKI twin (folder loki/)

In [ ]:
from loki_paths import TWIN_RUN_ROOT
FOLDER = f"{TWIN_RUN_ROOT}/data/loki/2026/3001/raw"     # the folder of your proposal (NICOS)

In [ ]:
import sys, os, json, glob, datetime, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath("../.."))
import numpy as np
import scipp as sc
import h5py
import matplotlib.pyplot as plt
from loki_paths import TWIN_RUN_ROOT
import loki_reduction as red
import loki_analysis as la
import loki_samples as ls
import nb_tools
nb_tools.need_measurements(FOLDER, "loki")   # stops here while there is no file

In [ ]:
rows = []
with open(f"{FOLDER}/log.jsonl", encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        d = entry["details"]
        if entry["action"] != "count" or not d.get("file"):
            continue
        s = d["state"]
        rows.append(dict(file=os.path.basename(d["file"]), kind=red.kind(s), title=d["title"], t_s=d["t"],
                         collimation=s["collimation"], pulse_skipping=s.get("pulse_skipping", "off"),
                         detector_m=s["detector_distance"], sample=s["sample"]))
def show_table(rows):                                  # a list of dicts, printed as a table
    if not rows:
        print("no measurement in the logbook"); return
    cols = list(rows[0])
    width = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in cols}
    print("  ".join(c.ljust(width[c]) for c in cols))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(width[c]) for c in cols))

show_table(rows)

## Your analysis

## Results for the committee

($I(Q)$, $R_g$, $R$, $\phi$, with uncertainties; logbook of your choices)

## Check your results (after writing them)

The twin knows the true sample: put your final numbers below and run the cell. A wrong result is explained.

In [ ]:
import loki_exercises as ex
rg, drg, i0 = None, None, None                     # Guinier [Å, Å, cm⁻¹]
radius, dradius, phi, dphi = None, None, None, None  # sphere model [Å, Å, -, -]
ex.check_guinier(rg, drg, i0)
ex.check_sphere(radius, dradius, phi, dphi)

## Submit your work

When you are done: save this notebook (*Save*), then submit. The file for your teacher is made from your measurements and your notebook.

In [ ]:
import submit
_ = submit.Submit("loki/exercises/silica_spheres", "beamtime")